# Distillation, step 0 — the students that write our text as it is

The companion of `Distill.ipynb` (roadmap §B, step 0), for the two students whose tokenizers
already cover Devanagari and Latin, so nothing is swapped: each is fine-tuned on the 30 h of
verified train labels and scored exactly as the transducers are, against the same Flex reference.
It needs its own runtime: `qwen-asr` pins transformers 4.57.6, and NeMo's kernel runs 5.x.

**Students.**
- **Qwen3-ASR-0.6B** (Alibaba, Apache-2.0): an audio encoder feeding a Qwen3 decoder, trained on 30
  languages including Hindi, not Nepali. Its prompt names the language, and `language None` means
  "no speech" to it, so the prompt is fixed at `language Nepali<asr_text>` and the loss is on the
  transcript alone. Chosen over the 1.7B because it is Parakeet's size and smaller than Flex; the
  1.7B is a one-line change if the 0.6B comes close.
- **Whisper-large-v3-turbo** (OpenAI, MIT): it knows Nepali (`<|ne|>`) but loops on it zero-shot
  (123% WER in the bake-off). 04a fine-tuned it to 14.62 on the 2026-09-12 gold; this re-runs it on
  the current export. Every clip is padded to 30 s, which is its encoder's only input length.

**Choices, fixed before any run.** Peak LR 2e-5 for Qwen (its official recipe) and 1e-5 for Whisper
(04a), linear decay after 10% warmup, up to 8 epochs (Whisper-turbo was still improving at 04a's 5),
stopping once val WER has gained less than 0.2 points over 3 epochs, a hand stop keeping the best
weights so far,
~12 min of audio per optimizer step, summed token cross-entropy divided by the step's tokens.
Decoding is greedy with Flex's loop retry: a clip whose output repeats a 3-word sequence 5+ times is
decoded again, alone, with a repetition penalty, no repeated 6-token phrase and a length cap from
its duration (the densest train label, in this model's tokens).

**Smoke-tested on CPU (2026-09-24, transformers 4.57.6).** Zero-shot, Qwen already writes rough
Nepanglish; 30 steps on four clips took its loss per token from 0.81 to 0.00, with the references
reproduced exactly, `।` and English included. The processor left-pads whatever the tokenizer says,
so the padding side is passed per call; with left padding the label mask had covered audio tokens.

**Keeping the A100 busy** (all in `ftkit.py`):
- **No disk I/O in the loop.** Audio sits in RAM as int16, and a clip is a slice of its episode.
- **Little padding.** Batches are built by duration and padded to whole seconds, so there are only
  ~20 distinct shapes and cudnn picks its kernels once per shape.
- **The CPU never stalls the GPU.** DataLoader workers collate and pin the next batches while the
  current one runs.
- **Measured batch size.** A probe finds the largest micro-batch that survives forward+backward on
  the longest clip, with the optimizer state already allocated. Training uses 90% of it, and
  gradient accumulation makes up the effective batch.
- **A100 arithmetic.** bf16 autocast, TF32 matmuls and fused AdamW.
- **Measured, not assumed.** The log reports throughput (× realtime), padding waste, GPU
  utilisation and memory every few steps. Single-digit utilisation or high padding waste means a
  setting needs changing.

## Config

In [ ]:
RUN_PREFIX = "distill-step0-2026-09-24"   # the same folder as Distill.ipynb's students
NOTEBOOK = "hf"                            # names this notebook's summary file
STUDENTS = ["qwen", "whisper"]
QWEN_ID = "Qwen/Qwen3-ASR-0.6B"
WHISPER_ID = "openai/whisper-large-v3-turbo"
QWEN_ASR_VERSION = "0.0.6"                 # pins transformers 4.57.6; the version smoke-tested on CPU
OUT_REPO = "Sagyam/nepanglish-asr-students"
FLEX_REPO = "Sagyam/nepanglish-asr-flex-ft"
FLEX_REFERENCE = "flex-xtalk-sweep-2026-09-22/flex-xtalk-sweep-2026-09-22-p00-s0"
# Per student: the most epochs, which is also the length of the linear LR decay. Whisper-turbo
# was still improving at 04a's 5; both keep their pretrained heads.
EPOCHS = {"qwen": 8, "whisper": 8}
PATIENCE, WARMUP = 3, 0.1
MIN_DELTA = 0.2           # val WER points 3 epochs must gain between them, or training stops
LR = {"qwen": 2e-5, "whisper": 1e-5}
EFFECTIVE_S = 720.0       # ~12 min of audio per optimizer step
PAD_TO_S = 1.0
PROBE_FRACTION = 0.9
EVAL_BUDGET_S, EVAL_ITEMS = 1200.0, 96
DEVICE = "cuda"
DATA_LOCAL = None         # a local export in the HF layout instead of the download

## Setup

In [ ]:
%pip install -q rapidfuzz "qwen-asr=={QWEN_ASR_VERSION}"
import json
import os
import sys
from pathlib import Path

import torch

IN_COLAB = "google.colab" in sys.modules
# Secrets only work from a cell run in the Colab UI. When cells are driven from outside (the Colab
# MCP), run this cell by hand once; the token is then kept in the hub's token file on the VM.
if IN_COLAB:
    from google.colab import userdata

    TOKEN_FILE = Path.home() / ".cache" / "huggingface" / "token"
    if not os.environ.get("HF_TOKEN"):
        os.environ["HF_TOKEN"] = (TOKEN_FILE.read_text().strip() if TOKEN_FILE.exists()
                                  else userdata.get("HF_TOKEN"))
    if not TOKEN_FILE.exists():
        TOKEN_FILE.parent.mkdir(parents=True, exist_ok=True)
        TOKEN_FILE.write_text(os.environ["HF_TOKEN"])
        TOKEN_FILE.chmod(0o600)
FT = Path("/content/ft") if IN_COLAB else Path.cwd() / ".cache-ft"
FT.mkdir(parents=True, exist_ok=True)
OUT_ROOT = FT / "out" / RUN_PREFIX
OUT_ROOT.mkdir(parents=True, exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
print("students:", STUDENTS, "| outputs:", OUT_ROOT)

In [ ]:
%%writefile /content/ft/ftkit.py
"""Shared fine-tuning kit for the Nepanglish ASR notebook (04c).

Everything that is not model-specific: the dataset held in RAM, duration-bucketed batches, the
harness scorer, a GPU utilisation monitor, a batch-size probe and one training loop. The notebook
writes this file out, so it imports exactly the same code as the main kernel. Keep it importable
on Python 3.10+ with numpy 1.x or 2.x.

How the GPU is kept busy:
  * audio is decoded once into RAM as int16; a clip is a slice, so no disk I/O in the loop;
  * batches are built by duration and padded to whole seconds, so there is little padding and
    only ~20 distinct shapes (cudnn.benchmark can then pick kernels once per shape);
  * DataLoader workers collate and pin the next batches while the GPU runs the current one;
  * a probe finds the largest micro-batch that survives forward+backward at the longest clip,
    with the optimizer state already allocated, and training uses a fixed fraction of it;
  * bf16 autocast, TF32 matmuls and fused AdamW; gradient accumulation reaches the effective
    batch; utilisation, throughput and padding waste are logged, not assumed.
"""

from __future__ import annotations

import json
import math
import random
import re
import shutil
import subprocess
import sys
import threading
import time
from collections import Counter
from collections.abc import Callable, Sequence
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any

import numpy as np
import soundfile as sf
import torch

REPO = "Sagyam/nepanglish-asr"
SR = 16_000


# --- data --------------------------------------------------------------------------------------


def download_dataset(local: str | None = None) -> Path:
    """The HF dataset snapshot, or a local copy in the same layout."""
    if local:
        return Path(local)
    from huggingface_hub import snapshot_download

    return Path(
        snapshot_download(
            REPO,
            repo_type="dataset",
            allow_patterns=["training/*", "gold/*", "harness/*", "analytics/*"],
        )
    )


def load_splits(data: Path) -> dict[str, list[dict]]:
    """train / val from the training export, gold from the gold export; asserts disjointness."""
    rows = [
        json.loads(line) for line in (data / "training" / "training.jsonl").open(encoding="utf-8")
    ]
    gold = [json.loads(line) for line in (data / "gold" / "gold.jsonl").open(encoding="utf-8")]
    splits = {
        "train": [r for r in rows if r["split"] == "train"],
        "val": [r for r in rows if r["split"] == "val"],
        "gold": gold,
    }
    trained = {r["segment_id"] for r in rows}
    assert not trained & {r["segment_id"] for r in gold}, "a gold clip is in the training export"
    return splits


def attach_speaker_turns(data: Path, rows: Sequence[dict]) -> int:
    """Copy each row's diarized turns, with their linked voices (D78, D87), from the analytics
    export: the training export does not carry them. Returns how many rows have turns."""
    want = {r["segment_id"]: r for r in rows}
    with (data / "analytics" / "analytics.jsonl").open(encoding="utf-8") as fh:
        for line in fh:
            a = json.loads(line)
            if a["segment_id"] in want:
                want[a["segment_id"]]["speaker_turns"] = a.get("speaker_turns")
    return sum(bool(r.get("speaker_turns")) for r in rows)


def duration(row: dict) -> float:
    return row["end_time"] - row["start_time"]


class AudioStore:
    """Every episode decoded once into RAM as int16; a clip is a slice of it (a view, no copy)."""

    def __init__(self, data: Path, episode_ids: Sequence[str]):
        self.audio: dict[str, np.ndarray] = {}
        for ep in sorted(set(episode_ids)):
            audio, sr = sf.read(data / "training" / "episodes" / f"{ep}.flac", dtype="int16")
            assert sr == SR and audio.ndim == 1, (ep, sr, audio.shape)
            self.audio[ep] = audio

    def clip(self, row: dict) -> np.ndarray:
        audio = self.audio[row["episode_id"]]
        return audio[round(row["start_time"] * SR) : round(row["end_time"] * SR)]

    def clip_f32(self, row: dict) -> torch.Tensor:
        return torch.from_numpy(self.clip(row).astype(np.float32) / 32768.0)

    @property
    def gib(self) -> float:
        return sum(a.nbytes for a in self.audio.values()) / 2**30


def bucket_batches(
    rows: Sequence[dict],
    *,
    budget_s: float,
    max_items: int,
    pad_to_s: float,
    shuffle: bool,
    seed: int = 0,
) -> list[list[int]]:
    """Index batches whose padded size (items x longest clip, rounded up to `pad_to_s`) fits
    `budget_s`. Sorting by duration keeps padding low; shuffling reorders whole batches and
    jitters lengths slightly so the same clips do not always share a batch."""
    rng = random.Random(seed)
    jitter = [rng.uniform(-0.3, 0.3) if shuffle else 0.0 for _ in rows]
    order = sorted(range(len(rows)), key=lambda i: duration(rows[i]) + jitter[i])
    batches: list[list[int]] = []
    cur: list[int] = []
    longest = 0.0
    for i in order:
        d = math.ceil(duration(rows[i]) / pad_to_s) * pad_to_s
        if cur and ((len(cur) + 1) * max(longest, d) > budget_s or len(cur) >= max_items):
            batches.append(cur)
            cur, longest = [], 0.0
        cur.append(i)
        longest = max(longest, d)
    if cur:
        batches.append(cur)
    if shuffle:
        rng.shuffle(batches)
    return batches


def pad_len(samples: int, pad_to_s: float) -> int:
    step = int(pad_to_s * SR)
    return math.ceil(samples / step) * step


class _Batches(torch.utils.data.Dataset):
    def __init__(self, rows, batches, collate):
        self.rows, self.batches, self.collate = rows, batches, collate

    def __len__(self):
        return len(self.batches)

    def __getitem__(self, i):
        return self.collate([self.rows[j] for j in self.batches[i]])


def loader(rows, batches, collate, workers: int) -> torch.utils.data.DataLoader:
    return torch.utils.data.DataLoader(
        _Batches(rows, batches, collate),
        batch_size=None,
        shuffle=False,
        num_workers=workers,
        pin_memory=True,
        prefetch_factor=4 if workers else None,
        persistent_workers=False,
    )


# --- scoring -----------------------------------------------------------------------------------


def is_loop(text: str) -> bool:
    """A 3-word sequence repeated 5 or more times: the decoder is stuck, not transcribing."""
    toks = text.split()
    top = Counter(zip(toks, toks[1:], toks[2:], strict=False)).most_common(1)
    return bool(top) and top[0][1] >= 5


class RetryLoops:
    """Wraps a batch `transcribe`: a clip whose first decode loops is decoded again, alone, by
    `retry` (anti-repetition settings); every other clip keeps its first decode untouched.
    `log` keeps (segment_id, first, retried), so the effect is measurable within one run."""

    def __init__(self, transcribe: Callable[[list[dict]], list[str]], retry: Callable[[dict], str]):
        self.transcribe, self.retry = transcribe, retry
        self.log: list[tuple[str, str, str]] = []

    def __call__(self, rows: list[dict]) -> list[str]:
        texts = self.transcribe(rows)
        for i, text in enumerate(texts):
            if is_loop(text):
                texts[i] = self.retry(rows[i])
                self.log.append((rows[i]["segment_id"], text, texts[i]))
        return texts


def harness_scorer(data: Path, work: Path) -> Callable[[Sequence[str], Sequence[str]], dict]:
    """fold.py from the dataset's harness/, imported from a real copy: the HF cache stores files as
    symlinks into its blob store, and normalize.py finds its config via its resolved path.

    Two layouts are accepted: the repo's (backend/app/services/, config/), and the flat one the
    2026-09-21 export uploaded (fold.py, normalize.py, normalization.yaml side by side), which
    is laid back out here because normalize.py looks for ../../../config/."""
    src, dst = data / "harness", work / "harness"
    if not dst.exists():
        if (src / "fold.py").exists():
            (dst / "backend" / "app" / "services").mkdir(parents=True)
            (dst / "config").mkdir()
            for name in ("fold.py", "normalize.py"):
                shutil.copyfile(src / name, dst / "backend" / "app" / "services" / name)
            shutil.copyfile(src / "normalization.yaml", dst / "config" / "normalization.yaml")
        else:
            shutil.copytree(src, dst)
    for name in [m for m in sys.modules if m == "app" or m.startswith("app.")]:
        del sys.modules[name]
    sys.path.insert(0, str(dst / "backend"))
    from app.services.fold import fold_tokens, word_errors
    from rapidfuzz.distance import Levenshtein

    dev = re.compile(r"[ऀ-ॿ]")

    def chars(text: str) -> str:
        return " ".join(t if dev.search(t) else t.lower() for t in fold_tokens(text))

    def score(refs: Sequence[str], hyps: Sequence[str]) -> dict:
        words = werr = rwords = rerr = nchars = cerr = loops = subs = dels = ins = 0
        for ref, hyp in zip(refs, hyps, strict=True):
            words += len(fold_tokens(ref))
            folded = word_errors(ref, hyp)
            werr += folded.errors
            subs, dels = subs + folded.substitutions, dels + folded.deletions
            ins += folded.insertions
            raw = word_errors(ref, hyp, folded=False)
            rerr, rwords = rerr + raw.errors, rwords + raw.ref_words
            nchars += len(chars(ref))
            cerr += Levenshtein.distance(chars(ref), chars(hyp))
            loops += is_loop(hyp)
        assert subs + dels + ins == werr, "S + D + I must add up to the folded errors"
        return {
            "wer": 100 * werr / max(words, 1),
            "raw_wer": 100 * rerr / max(rwords, 1),
            "cer": 100 * cerr / max(nchars, 1),
            # folded, per 100 reference words. In crosstalk a label keeps what was audible, which is
            # not always both voices, so a model that writes the other one can be charged insertions
            "sub": 100 * subs / max(words, 1),
            "del": 100 * dels / max(words, 1),
            "ins": 100 * ins / max(words, 1),
            "loops": loops,
            "clips": len(refs),
        }

    def per_clip(refs: Sequence[str], hyps: Sequence[str]) -> list[dict]:
        """Folded counts per clip, for paired comparisons: errors = sub + del + ins."""
        out = []
        for ref, hyp in zip(refs, hyps, strict=True):
            a = word_errors(ref, hyp)
            out.append(
                {
                    "words": len(fold_tokens(ref)),
                    "errors": a.errors,
                    "sub": a.substitutions,
                    "del": a.deletions,
                    "ins": a.insertions,
                }
            )
        return out

    score.per_clip = per_clip
    return score


def transcribe_rows(
    rows: Sequence[dict],
    transcribe: Callable[[list[dict]], list[str]],
    *,
    budget_s: float,
    max_items: int,
    pad_to_s: float,
) -> tuple[list[str], list[float]]:
    """Run `transcribe` over duration-bucketed batches; returns texts and per-clip compute seconds
    in the original row order."""
    texts: list[str] = [""] * len(rows)
    compute = [0.0] * len(rows)
    for batch in bucket_batches(
        rows, budget_s=budget_s, max_items=max_items, pad_to_s=pad_to_s, shuffle=False
    ):
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        out = transcribe([rows[i] for i in batch])
        torch.cuda.synchronize()
        took = time.perf_counter() - t0
        audio_s = sum(duration(rows[i]) for i in batch)
        for i, text in zip(batch, out, strict=True):
            texts[i] = text.strip()
            compute[i] = took * duration(rows[i]) / audio_s
    return texts, compute


def write_hyps(
    path: Path, rows: Sequence[dict], texts: Sequence[str], compute: Sequence[float]
) -> None:
    """The hypothesis-cache format the bake-off comparisons used, so the result can join them."""
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as fh:
        for r, text, c in zip(rows, texts, compute, strict=True):
            fh.write(
                json.dumps(
                    {"segment_id": r["segment_id"], "text": text, "compute_s": c},
                    ensure_ascii=False,
                )
                + "\n"
            )


# --- GPU ---------------------------------------------------------------------------------------


def fast_cuda() -> None:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True  # safe: shapes are padded to whole seconds
    torch.set_float32_matmul_precision("high")


class GpuMonitor:
    """Samples nvidia-smi in a background thread; `window()` returns the mean utilisation and the
    peak memory since the previous call."""

    def __init__(self, every: float = 1.0):
        self.every, self._util, self._mem = every, [], []
        self._stop = threading.Event()
        self._thread = threading.Thread(target=self._run, daemon=True)

    def _run(self):
        while not self._stop.is_set():
            try:
                out = subprocess.run(
                    [
                        "nvidia-smi",
                        "--query-gpu=utilization.gpu,memory.used",
                        "--format=csv,noheader,nounits",
                    ],
                    capture_output=True,
                    text=True,
                    timeout=5,
                )
                util, mem = (float(x) for x in out.stdout.strip().splitlines()[0].split(","))
                self._util.append(util)
                self._mem.append(mem)
            except Exception:
                pass
            self._stop.wait(self.every)

    def start(self) -> GpuMonitor:
        self._thread.start()
        return self

    def stop(self) -> None:
        self._stop.set()

    def window(self) -> dict:
        util, mem = self._util, self._mem
        self._util, self._mem = [], []
        return {
            "gpu_util": float(np.mean(util)) if util else float("nan"),
            "gpu_mem_gib": max(mem) / 1024 if mem else float("nan"),
        }


def init_optimizer_state(model: torch.nn.Module, optimizer: torch.optim.Optimizer) -> None:
    """Allocate AdamW's moment buffers before probing, without moving any weight: with zero
    gradients and zero weight decay an AdamW step is a no-op, but it creates the state."""
    decay = [g["weight_decay"] for g in optimizer.param_groups]
    for g in optimizer.param_groups:
        g["weight_decay"] = 0.0
    for p in model.parameters():
        if p.requires_grad:
            p.grad = torch.zeros_like(p)
    optimizer.step()
    optimizer.zero_grad(set_to_none=True)
    for g, d in zip(optimizer.param_groups, decay, strict=True):
        g["weight_decay"] = d
    for state in optimizer.state.values():
        if "step" in state:
            state["step"].zero_()


def probe_max_items(
    step: Callable[[int], None], lo: int, hi: int, params: Sequence[torch.nn.Parameter]
) -> int:
    """The largest n in [lo, hi] for which `step(n)` (one forward+backward at the worst case)
    fits in memory. `step` must not touch the gradients.

    The gradient buffer stays allocated throughout, as it does in training from the second
    micro-batch of every accumulated step: freeing it between probes measured the activations
    without it and picked a batch that ran out of memory once training accumulated."""
    for p in params:
        p.grad = torch.zeros_like(p)
    best = 0
    while lo <= hi:
        mid = (lo + hi) // 2
        try:
            step(mid)
            torch.cuda.synchronize()
            best, lo = mid, mid + 1
        except torch.cuda.OutOfMemoryError:
            hi = mid - 1
        for p in params:
            p.grad.zero_()
        torch.cuda.empty_cache()
    for p in params:
        p.grad = None
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    return best


# --- training ----------------------------------------------------------------------------------


@dataclass
class TrainConfig:
    name: str
    out: str
    epochs: int = 10
    lr: float = 1e-5
    schedule: str = "linear"  # linear warmup+decay, or "tristage" (fairseq2's default)
    warmup_frac: float = 0.1
    effective_s: float = 720.0  # audio seconds per optimizer step, reached by accumulation
    weight_decay: float = 0.0
    clip_norm: float = 1.0
    patience: int = 3  # evaluations without a val-WER improvement before stopping
    # WER points an evaluation must gain on the last counted one to reset `patience`. 0 counts
    # any gain, so a crawl of 0.2 points an epoch never stops. The best weights are kept either way.
    min_delta: float = 0.0
    seed: int = 0
    log_every: int = 10
    workers: int = 6


def lr_factor(step: int, total: int, cfg: TrainConfig) -> float:
    warm = max(1, int(cfg.warmup_frac * total))
    if step < warm:
        return (step + 1) / warm
    if cfg.schedule == "tristage":  # 10% warmup, 40% hold, 50% exponential decay to 5%
        hold_end = int(0.5 * total)
        if step < hold_end:
            return 1.0
        frac = (step - hold_end) / max(1, total - hold_end)
        return math.exp(math.log(0.05) * frac)
    return max(0.0, (total - step) / max(1, total - warm))


def group_steps(rows, batches, effective_s: float) -> list[list[list[int]]]:
    """Micro-batches grouped into optimizer steps of about `effective_s` seconds of real audio."""
    steps, cur, acc = [], [], 0.0
    for b in batches:
        cur.append(b)
        acc += sum(duration(rows[i]) for i in b)
        if acc >= effective_s:
            steps.append(cur)
            cur, acc = [], 0.0
    if cur:
        steps.append(cur)
    return steps


def sid(m: dict) -> str:
    """A score's error split, per 100 reference words: S + D + I is its folded WER. Read all three
    on crosstalk: the labels drop the other voice, so hearing it costs insertions."""
    return f"S {m['sub']:.2f}  D {m['del']:.2f}  I {m['ins']:.2f}"


def retry_note(val: dict) -> str:
    """For an `evaluate` that wraps its decode in RetryLoops and reports the greedy score too."""
    if "retried" not in val:
        return ""
    return f"  (greedy WER {val['greedy_wer']:.2f}, {val['retried']} retried)"


def speed_check(
    model: torch.nn.Module,
    *,
    rows: Sequence[dict],
    batches: list[list[int]],
    collate: Callable[[list[dict]], Any],
    loss_fn: Callable[[torch.nn.Module, Any], tuple[torch.Tensor, int]],
    evaluate: Callable[[torch.nn.Module], dict],
    val_rows: Sequence[dict],
    gold_rows: Sequence[dict],
    epochs: int,
    monitor: GpuMonitor,
    workers: int = 6,
    n_micro: int = 10,
) -> dict:
    """Time the run before committing to it: forward+backward on `n_micro` real training
    micro-batches (no optimizer step, so no weight moves), then one full val pass, which is also
    the val WER before training. Projects the wall time of every epoch and the gold pass.

    The micro-batches run twice and only the second pass is timed, so cudnn's per-shape kernel
    search and worker start-up are excluded. Gradients stay allocated between micro-batches, as
    they do under accumulation, so the memory reading is training's. BatchNorm running
    statistics are restored after."""
    bns = [m for m in model.modules() if isinstance(m, torch.nn.modules.batchnorm._BatchNorm)]
    saved = [{k: v.clone() for k, v in m.state_dict().items()} for m in bns]
    sample = batches[:n_micro]
    it = iter(loader(rows, sample + sample, collate, workers))

    def run() -> tuple[float, float, int]:
        audio = padded = 0.0
        clips = 0
        for _ in sample:
            b = next(it)
            with torch.autocast("cuda", dtype=torch.bfloat16):
                loss, _ = loss_fn(model, b)
            loss.backward()
            model.zero_grad(set_to_none=False)
            audio, padded = audio + b["seconds"], padded + b["padded_seconds"]
            clips += len(b["lens"]) if "lens" in b else b["wav"].shape[0]
        torch.cuda.synchronize()
        return audio, padded, clips

    model.train()
    run()
    monitor.window()
    t0 = time.perf_counter()
    audio, padded, clips = run()
    train_dt = time.perf_counter() - t0
    gpu = monitor.window()
    model.zero_grad(set_to_none=True)
    for m, s in zip(bns, saved, strict=True):
        m.load_state_dict(s)

    model.eval()
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        val = evaluate(model)
    torch.cuda.synchronize()
    val_dt = time.perf_counter() - t0
    model.train()

    secs = lambda rs: sum(duration(r) for r in rs)  # noqa: E731
    epoch_s = secs(rows) / (audio / train_dt)
    gold_s = val_dt * secs(gold_rows) / secs(val_rows)
    rec = {
        "train_x_realtime": audio / train_dt,
        "train_ms_per_clip": 1000 * train_dt / clips,
        "padding_waste": 1 - audio / padded,
        **gpu,
        "val_s": val_dt,
        "val_ms_per_clip": 1000 * val_dt / len(val_rows),
        **{f"val_before_{k}": v for k, v in val.items()},
        "epoch_min": epoch_s / 60,
        "projected_h": (epochs * (epoch_s + val_dt) + gold_s) / 3600,
    }
    print(
        f"train: {rec['train_x_realtime']:.0f}x realtime = "
        f"{rec['train_ms_per_clip']:.0f} ms per clip "
        f"(forward+backward, {clips} clips), pad waste {rec['padding_waste']:.0%}, "
        f"GPU {rec['gpu_util']:.0f}% util, {rec['gpu_mem_gib']:.1f} GiB\n"
        f"val:   {val_dt:.0f} s for {len(val_rows)} clips = "
        f"{rec['val_ms_per_clip']:.0f} ms per clip "
        f"(batched decode); WER before training {val['wer']:.2f} ({sid(val)}), loops {val['loops']}"
        f"{retry_note(val)}\n"
        f"projected: {rec['epoch_min']:.1f} min per epoch + {val_dt / 60:.1f} min val -> "
        f"at most {rec['projected_h']:.2f} h for {epochs} epochs and gold "
        "(early stopping can cut it)",
        flush=True,
    )
    return rec


def train(
    model: torch.nn.Module,
    *,
    cfg: TrainConfig,
    rows: Sequence[dict],
    make_batches: Callable[[int], list[list[int]]],
    collate: Callable[[list[dict]], Any],
    loss_fn: Callable[[torch.nn.Module, Any], tuple[torch.Tensor, int]],
    evaluate: Callable[[torch.nn.Module], dict],
    save_best: Callable[[torch.nn.Module], None],
    optimizer: torch.optim.Optimizer,
    monitor: GpuMonitor,
) -> dict:
    """Train with bf16 autocast and gradient accumulation; evaluate on val after every epoch;
    keep the best weights (by val WER) in CPU memory and hand them to `save_best`. A hand stop
    (KeyboardInterrupt) after the first evaluation ends training the same way.

    `loss_fn` returns a *summed* loss and the number of units it sums over (clips for CTC, tokens
    for cross-entropy); gradients are divided by the step's total units, so accumulated
    micro-batches of different sizes are weighted exactly. Count the units on the CPU (in
    `collate`): an `int()` of a GPU tensor stalls the host once per micro-batch."""
    out = Path(cfg.out)
    out.mkdir(parents=True, exist_ok=True)
    torch.manual_seed(cfg.seed)
    plan = [group_steps(rows, make_batches(e), cfg.effective_s) for e in range(cfg.epochs)]
    total = sum(len(p) for p in plan)
    base_lrs = [g["lr"] for g in optimizer.param_groups]
    params = [p for p in model.parameters() if p.requires_grad]
    history, best, best_state, bad, step = [], float("inf"), None, 0, 0
    counted = float("inf")  # the val WER that last reset `bad`
    print(
        f"{cfg.name}: {total} optimizer steps over {cfg.epochs} epochs "
        f"(~{cfg.effective_s / 60:.0f} min of audio each), peak lr {cfg.lr:g}, {cfg.schedule}"
    )
    stopped_by_hand = False
    try:
        for epoch in range(cfg.epochs):
            model.train()
            flat = [b for s in plan[epoch] for b in s]
            sizes = [len(s) for s in plan[epoch]]
            it = iter(loader(rows, flat, collate, cfg.workers))
            t_log, audio_log, padded_log, loss_log, units_log = (
                time.perf_counter(),
                0.0,
                0.0,
                0.0,
                0,
            )
            monitor.window()
            for n_micro in sizes:
                for g, lr0 in zip(optimizer.param_groups, base_lrs, strict=True):
                    g["lr"] = lr0 * lr_factor(step, total, cfg)
                step_units = 0
                for _ in range(n_micro):
                    batch = next(it)
                    with torch.autocast("cuda", dtype=torch.bfloat16):
                        loss, units = loss_fn(model, batch)
                    loss.backward()
                    step_units += units
                    loss_log += loss.detach()  # stays on the GPU: no host sync per micro-batch
                    units_log += units
                    audio_log += batch["seconds"]
                    padded_log += batch["padded_seconds"]
                torch._foreach_div_(
                    [p.grad for p in params if p.grad is not None], max(step_units, 1)
                )
                gnorm = torch.nn.utils.clip_grad_norm_(params, cfg.clip_norm)
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)
                step += 1
                if step % cfg.log_every == 0:
                    torch.cuda.synchronize()
                    dt = time.perf_counter() - t_log
                    gpu = monitor.window()
                    rec = {
                        "step": step,
                        "epoch": epoch + 1,
                        "lr": optimizer.param_groups[0]["lr"],
                        "loss": float(loss_log) / max(units_log, 1),
                        "grad_norm": float(gnorm),
                        "audio_x_realtime": audio_log / dt,
                        "padding_waste": 1 - audio_log / max(padded_log, 1e-9),
                        "peak_alloc_gib": torch.cuda.max_memory_allocated() / 2**30,
                        **gpu,
                    }
                    history.append(rec)
                    print(
                        f"step {step:5d}/{total} ep {epoch + 1} lr {rec['lr']:.2e} "
                        f"loss {rec['loss']:.4f} | {rec['audio_x_realtime']:6.0f}x realtime, "
                        f"pad waste {rec['padding_waste']:.0%}, "
                        f"GPU {rec['gpu_util']:.0f}% util, {rec['gpu_mem_gib']:.1f} GiB",
                        flush=True,
                    )
                    t_log, audio_log, padded_log, loss_log, units_log = (
                        time.perf_counter(),
                        0.0,
                        0.0,
                        0.0,
                        0,
                    )
            model.eval()
            with torch.no_grad():
                val = evaluate(model)
            history.append(
                {"epoch": epoch + 1, "step": step, **{f"val_{k}": v for k, v in val.items()}}
            )
            improved = val["wer"] < best
            print(
                f"== epoch {epoch + 1}: val WER {val['wer']:.2f} ({sid(val)})  "
                f"CER {val['cer']:.2f}  raw WER {val['raw_wer']:.2f}  "
                f"loops {val['loops']}{retry_note(val)}" + ("  (best)" if improved else ""),
                flush=True,
            )
            (out / "history.json").write_text(json.dumps(history, indent=1))
            if improved:
                best_state = {
                    k: v.detach().to("cpu", copy=True) for k, v in model.state_dict().items()
                }
                best = val["wer"]
            if val["wer"] < counted - cfg.min_delta:  # at 0: the old rule, any strict gain
                counted, bad = val["wer"], 0
            else:
                bad += 1
                if bad >= cfg.patience:
                    print(
                        f"val WER gained less than {cfg.min_delta:g} points in {cfg.patience} "
                        "evaluations; stopping"
                    )
                    break
    except KeyboardInterrupt:
        # Colab's stop button: end training here and keep the best weights so far, so they are
        # saved and scored like a finished run's. Before any evaluation there are none to keep.
        if best_state is None:
            raise
        stopped_by_hand = True
        print(f"stopped by hand; keeping the best weights (val WER {best:.2f})", flush=True)
    monitor.window()
    if best_state is not None:
        model.load_state_dict(best_state)
    model.eval()
    save_best(model)
    (out / "config.json").write_text(json.dumps(asdict(cfg), indent=1))
    return {
        "best_val_wer": best,
        "steps": step,
        "history": history,
        "stopped_by_hand": stopped_by_hand,
    }

In [ ]:
%%writefile /content/ft/sweep.py
"""The crosstalk sweep (D96): which run's weights are kept, and how two runs are compared.

The notebook trains one model per (XTALK_P, seed) point on the same export. The winner is chosen by
val WER alone, under a rule fixed before any sweep result was read: augmentation has to beat p = 0
by more than the seed noise, measured as the gap between the two p = 0 seeds, or p = 0 is kept.
Gold never takes part in the choice, so every gold number stays a held-out score.

Runs are compared on gold with a paired bootstrap that resamples whole episodes: clips of one
episode share a room and voices, and resampling them one by one would pretend to more independent
evidence than gold holds (findings.md, *How big gold has to be*). Pure numpy; importable without
torch.
"""

from __future__ import annotations

from collections.abc import Sequence

import numpy as np


def run_name(prefix: str, p: float, seed: int) -> str:
    """`<prefix>-p30-s0` for XTALK_P = 0.3 and seed 0: the run's folder and Models page id."""
    return f"{prefix}-p{round(p * 100):02d}-s{seed}"


def choose_winner(rows: Sequence[dict]) -> tuple[dict, str]:
    """The run whose weights are kept, and why, from rows with `xtalk_p`, `seed` and `val_wer`.

    p = 0 is its better seed. The noise is the gap between its two best seeds; with one p = 0 seed
    it is unmeasured and the rule is strict. The best augmented run wins only if it beats p = 0 by
    more than the noise; a tie goes to p = 0, the simpler recipe."""
    if not rows:
        raise ValueError("no runs to choose from")
    base = sorted((r for r in rows if r["xtalk_p"] == 0), key=lambda r: r["val_wer"])
    aug = sorted((r for r in rows if r["xtalk_p"] != 0), key=lambda r: r["val_wer"])
    if not base or not aug:
        best = (base or aug)[0]
        return best, "only one kind of run, so the lowest val WER"
    if len(base) >= 2:
        noise = base[1]["val_wer"] - base[0]["val_wer"]
        noise_note = f"seed noise {noise:.2f} (p=0 seeds {base[0]['seed']} and {base[1]['seed']})"
    else:
        noise, noise_note = 0.0, "seed noise unmeasured (one p=0 seed), so the rule is strict"
    margin = base[0]["val_wer"] - aug[0]["val_wer"]
    if margin > noise:
        return aug[0], f"p={aug[0]['xtalk_p']} beats p=0 on val by {margin:.2f} > {noise_note}"
    return base[0], f"best augmented margin {margin:.2f} is within {noise_note}; p=0 kept"


def paired_bootstrap(
    a: Sequence[dict],
    b: Sequence[dict],
    episodes: Sequence[str],
    *,
    n: int = 2000,
    seed: int = 0,
) -> tuple[float, float, float]:
    """WER(b) - WER(a) in points, with a 95% interval from resampling episodes.

    `a` and `b` are per-clip `{"errors", "words"}` for the same clips in the same order, and
    `episodes` names each clip's episode. WER is pooled (sum of errors over sum of words)."""
    if not (len(a) == len(b) == len(episodes)):
        raise ValueError("a, b and episodes must describe the same clips")
    ids = sorted(set(episodes))
    index = {e: i for i, e in enumerate(ids)}
    ep = np.array([index[e] for e in episodes])
    per = np.zeros((len(ids), 3))  # errors of a, errors of b, reference words
    clips = [[x["errors"], y["errors"], x["words"]] for x, y in zip(a, b, strict=True)]
    np.add.at(per, ep, np.array(clips))

    def diff(t: np.ndarray) -> float:
        return float(100 * (t[1] - t[0]) / max(t[2], 1))

    point = diff(per.sum(axis=0))
    rng = np.random.default_rng(seed)
    draws = rng.integers(0, len(ids), size=(n, len(ids)))
    totals = per[draws].sum(axis=1)  # (n, 3)
    boot = 100 * (totals[:, 1] - totals[:, 0]) / np.maximum(totals[:, 2], 1)
    lo, hi = np.percentile(boot, [2.5, 97.5])
    return point, float(lo), float(hi)

In [ ]:
%%writefile /content/ft/distill.py
"""Pure helpers for the distillation notebook (roadmap §B).

No torch and no NeMo, so the rules that decide what shapes the students' tokenizer and what their
scores are paired against are tested from backend/tests. Keep it importable on Python 3.10+.
"""

from __future__ import annotations

from collections.abc import Callable, Mapping, Sequence
from typing import Any


def tokenizer_texts(splits: Mapping[str, Sequence[dict]]) -> list[str]:
    """The labels the shared student tokenizer is trained on: train only.

    Val and gold text never shapes the vocabulary, so a word only they contain is scored as the
    student would meet it on new audio. Raises if a val or gold clip is in train."""
    held = {r["segment_id"] for name in ("val", "gold") for r in splits.get(name, ())}
    leaked = sorted(held & {r["segment_id"] for r in splits["train"]})
    if leaked:
        raise ValueError(f"held-out clips in train: {leaked[:5]}")
    return [r["text"] for r in splits["train"]]


def reference_texts(rows: Sequence[dict], by_id: Mapping[str, str]) -> list[str]:
    """Another system's transcripts in `rows`' order, for pairing clip by clip. Raises, naming
    them, if any row has none: a pairing over a subset would not be the same clips."""
    missing = [r["segment_id"] for r in rows if r["segment_id"] not in by_id]
    if missing:
        raise ValueError(f"{len(missing)} clip(s) have no reference transcript: {missing[:5]}")
    return [by_id[r["segment_id"]] for r in rows]


def by_class(
    rows: Sequence[dict],
    refs: Sequence[str],
    hyps: Sequence[str],
    score: Callable[[Sequence[str], Sequence[str]], Any],
    keys: Sequence[str] | None = None,
) -> dict[str, dict[str, Any]]:
    """`score` per value of each clip class (D87), as {key: {value: score}}.

    `keys` defaults to every class key any row carries. A clip without a key (or with no classes)
    is left out of that key's groups, not counted under a made-up value."""
    if not (len(rows) == len(refs) == len(hyps)):
        raise ValueError("rows, refs and hyps must describe the same clips")
    if keys is None:
        keys = sorted({k for r in rows for k in (r.get("classes") or {})})
    out: dict[str, dict[str, Any]] = {}
    for key in keys:
        groups: dict[str, list[int]] = {}
        for i, r in enumerate(rows):
            value = (r.get("classes") or {}).get(key)
            if value is not None:
                groups.setdefault(str(value), []).append(i)
        if groups:
            out[key] = {
                v: score([refs[i] for i in idx], [hyps[i] for i in idx])
                for v, idx in sorted(groups.items())
            }
    return out

In [ ]:
sys.path.insert(0, str(FT))
import distill
import ftkit
import sweep

ftkit.fast_cuda()
DATA = ftkit.download_dataset(DATA_LOCAL)
splits = ftkit.load_splits(DATA)
store = ftkit.AudioStore(DATA, [r["episode_id"] for rows in splits.values() for r in rows])
score = ftkit.harness_scorer(DATA, FT)
export = json.loads((DATA / "training" / "manifest.json").read_text())
print({k: len(v) for k, v in splits.items()}, f"audio in RAM: {store.gib:.1f} GiB |",
      "export", export.get("exported_at"))

## Flex, the reference

Flex's transcripts of this export's val and gold, rescored here. The numbers should match its card
(val 7.19, gold 11.56 on the 2026-09-22 export); a clip relabelled since moves them slightly.

In [ ]:
from huggingface_hub import HfApi, hf_hub_download

BUCKETS = ("none", "0-5%", "5-15%", ">15%")
REPORT_KEYS = ("overlap", "snr", "speakers", "cmi", "duration")  # also paired against Flex


def read_hyps(path):
    return {j["segment_id"]: j["text"] for j in map(json.loads, Path(path).open(encoding="utf-8"))}


flex = {}
for name in ("val", "gold"):
    path = hf_hub_download(FLEX_REPO, f"{FLEX_REFERENCE}/harness/{name}.jsonl", token=os.environ["HF_TOKEN"])
    flex[name] = distill.reference_texts(splits[name], read_hyps(path))
flex_scores = {name: score([r["text"] for r in splits[name]], flex[name]) for name in flex}
for name, m in flex_scores.items():
    print(f"Flex {name}: WER {m['wer']:.2f} ({ftkit.sid(m)}), raw {m['raw_wer']:.2f}, CER {m['cer']:.2f}")

## The students: loading, loss and decoding

Everything model-specific, one set of functions per student. `load_student` points `collate`,
`loss_fn` and `transcribe` at the right one, so ftkit's loop and the scoring cells are shared.

In [ ]:
import math
import shutil
import warnings

import numpy as np
import transformers
from huggingface_hub import snapshot_download
from qwen_asr import Qwen3ASRModel
from transformers import WhisperForConditionalGeneration, WhisperProcessor

transformers.logging.set_verbosity_error()
warnings.filterwarnings("ignore", module="transformers")

# The recogniser is told the language in its prompt. Nepali is not one of its 30, and
# "language None" means "no speech" to it, so the prompt names Nepali and the loss is on the
# transcript alone: the tag is a fixed prompt, not something the model has to learn to say.
QWEN_LANGUAGE = "Nepali"
ARCHITECTURE = {
    "qwen": "Qwen3-ASR: audio encoder + Qwen3 decoder, 0.6B; byte-level BPE, no vocabulary change",
    "whisper": "Whisper-large-v3-turbo: 32L encoder + 4L decoder, 0.8B; no vocabulary change",
}
BASE_MODEL = {"qwen": QWEN_ID, "whisper": WHISPER_ID}
CARD_NOTE = "its own tokenizer"
DECODER_NAME = "greedy+retry"


def lr_card(name):
    return LR[name]


def floats(rows):
    return [store.clip(r).astype(np.float32) / 32768.0 for r in rows]


# --- Qwen3-ASR ---------------------------------------------------------------------------------


def load_qwen():
    global model, processor, PROMPT
    wrapper = Qwen3ASRModel.from_pretrained(QWEN_ID, dtype=torch.float32, device_map=None)
    model, processor = wrapper.model.to(DEVICE), wrapper.processor
    # The shipped generation config sets a sampling temperature alongside greedy decoding, which
    # transformers refuses to save; decoding here is always greedy.
    for key in ("temperature", "top_p", "top_k"):
        setattr(model.generation_config, key, None)
    msgs = [{"role": "system", "content": ""}, {"role": "user", "content": [{"type": "audio", "audio": ""}]}]
    PROMPT = (processor.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)
              + f"language {QWEN_LANGUAGE}<asr_text>")
    return model


def _qwen_inputs(clips, texts, side):
    # the processor ignores tokenizer.padding_side; it has to be passed per call
    return processor(text=texts, audio=clips, return_tensors="pt", padding=True, padding_side=side)


def qwen_collate(rows):
    """Prompt + transcript + EOS, right-padded; labels only on the transcript and EOS."""
    clips = floats(rows)
    eos = processor.tokenizer.eos_token
    full = _qwen_inputs(clips, [PROMPT + r["text"] + eos for r in rows], "right")
    prefix = _qwen_inputs(clips, [PROMPT] * len(rows), "right")
    labels = full["input_ids"].clone()
    for i, n in enumerate(prefix["attention_mask"].sum(dim=1).tolist()):
        labels[i, :n] = -100
    labels[full["attention_mask"] == 0] = -100
    return {**full, "labels": labels, "tokens": int((labels[:, 1:] != -100).sum()),
            "lens": torch.tensor([len(c) for c in clips]), "seconds": sum(ftkit.duration(r) for r in rows),
            "padded_seconds": len(rows) * max(len(c) for c in clips) / ftkit.SR}


QWEN_KEYS = ("input_ids", "attention_mask", "input_features", "feature_attention_mask")


def qwen_loss(model, b):
    """Summed token cross-entropy: HF's mean over the label tokens times their count."""
    kw = {k: b[k].to(DEVICE, non_blocking=True) for k in QWEN_KEYS}
    out = model.thinker(**kw, labels=b["labels"].to(DEVICE, non_blocking=True))
    return out.loss * b["tokens"], b["tokens"]


def qwen_transcribe(rows, **gen):
    inputs = _qwen_inputs(floats(rows), [PROMPT] * len(rows), "left").to(DEVICE)
    with torch.no_grad(), torch.autocast(DEVICE, dtype=torch.bfloat16):
        out = model.generate(**inputs, do_sample=False, num_beams=1, **{"max_new_tokens": MAX_NEW_TOKENS, **gen})
    seqs = out.sequences if hasattr(out, "sequences") else out
    return processor.batch_decode(seqs[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True,
                                  clean_up_tokenization_spaces=False)


def qwen_tokens(text):
    return len(processor.tokenizer(text, add_special_tokens=False).input_ids)


def save_qwen(model, dst):
    model.save_pretrained(dst, state_dict={k: v.to(torch.bfloat16) for k, v in model.state_dict().items()})
    processor.save_pretrained(dst)
    base = Path(snapshot_download(QWEN_ID))  # the files qwen-asr needs to load a checkpoint
    for f in base.iterdir():
        if f.suffix in {".json", ".txt"} and not (dst / f.name).exists():
            shutil.copy(f, dst / f.name)


# --- Whisper -----------------------------------------------------------------------------------


def load_whisper():
    global model, processor, PREFIX, EOT
    processor = WhisperProcessor.from_pretrained(WHISPER_ID)
    model = WhisperForConditionalGeneration.from_pretrained(WHISPER_ID, torch_dtype=torch.float32).to(DEVICE)
    model.generation_config.forced_decoder_ids = None
    tok = processor.tokenizer
    PREFIX = tok.convert_tokens_to_ids(["<|startoftranscript|>", "<|ne|>", "<|transcribe|>", "<|notimestamps|>"])
    EOT = tok.eos_token_id
    return model


def whisper_collate(rows):
    """Log-mel features, every clip padded to 30 s (the encoder takes nothing else), and the prefix
    + transcript + end-of-text, with loss on the transcript and end-of-text only."""
    clips = floats(rows)
    feats = processor.feature_extractor(clips, sampling_rate=ftkit.SR, return_tensors="pt").input_features
    ids = [PREFIX + processor.tokenizer(r["text"], add_special_tokens=False).input_ids + [EOT] for r in rows]
    width = max(len(i) for i in ids) - 1
    inp = torch.full((len(rows), width), EOT, dtype=torch.long)
    lab = torch.full((len(rows), width), -100, dtype=torch.long)
    for i, full in enumerate(ids):
        inp[i, :len(full) - 1] = torch.tensor(full[:-1])
        lab[i, len(PREFIX) - 1:len(full) - 1] = torch.tensor(full[len(PREFIX):])
    return {"input_features": feats, "decoder_input_ids": inp, "labels": lab, "tokens": int((lab != -100).sum()),
            "lens": torch.tensor([len(c) for c in clips]), "seconds": sum(ftkit.duration(r) for r in rows),
            "padded_seconds": 30.0 * len(rows)}


def whisper_loss(model, b):
    logits = model(input_features=b["input_features"].to(DEVICE, non_blocking=True),
                   decoder_input_ids=b["decoder_input_ids"].to(DEVICE, non_blocking=True)).logits
    loss = torch.nn.functional.cross_entropy(logits.float().flatten(0, 1),
                                             b["labels"].to(DEVICE, non_blocking=True).flatten(),
                                             ignore_index=-100, reduction="sum")
    return loss, b["tokens"]


def whisper_transcribe(rows, **gen):
    feats = processor.feature_extractor(floats(rows), sampling_rate=ftkit.SR, return_tensors="pt").input_features
    with torch.no_grad(), torch.autocast(DEVICE, dtype=torch.bfloat16):
        out = model.generate(input_features=feats.to(DEVICE), language="ne", task="transcribe",
                             do_sample=False, num_beams=1, **{"max_new_tokens": MAX_NEW_TOKENS, **gen})
    return processor.batch_decode(out, skip_special_tokens=True)


def whisper_tokens(text):
    return len(processor.tokenizer(text, add_special_tokens=False).input_ids)


def save_whisper(model, dst):
    model.save_pretrained(dst, state_dict={k: v.to(torch.bfloat16) for k, v in model.state_dict().items()})
    processor.save_pretrained(dst)


# --- dispatch and the loop retry ---------------------------------------------------------------

KIT = {
    "qwen": (load_qwen, qwen_collate, qwen_loss, qwen_transcribe, qwen_tokens, save_qwen),
    "whisper": (load_whisper, whisper_collate, whisper_loss, whisper_transcribe, whisper_tokens, save_whisper),
}
RETRY = {"repetition_penalty": 1.2, "no_repeat_ngram_size": 6}
# Output caps, measured on the labels (2026-09-22 export). Both tokenizers are byte-level and spend
# several tokens per Devanagari character: gold runs to 452 Qwen tokens and 518 Whisper tokens.
# Whisper's decoder stops at 448 positions, 4 of them the prefix, so 5 gold clips cannot be written
# whole by it in one pass; that is the architecture, and it stays in the score.
MAX_NEW = {"qwen": 600, "whisper": 444}


def load_student(name):
    """A fresh student on DEVICE; points collate, loss_fn, transcribe and save_to at its kit, and
    measures its densest train label in its own tokens for the retry's length cap."""
    global collate, loss_fn, transcribe, save_to, MAX_TOKENS_PER_S, MAX_NEW_TOKENS
    load, collate, loss_fn, transcribe, count, save_to = KIT[name]
    load()
    MAX_NEW_TOKENS = MAX_NEW[name]
    before = len(splits["train"])
    splits["train"] = [r for r in splits["train"] if count(r["text"]) + 1 <= MAX_NEW_TOKENS]
    print(f"{name}: dropped {before - len(splits['train'])} train clip(s) longer than {MAX_NEW_TOKENS} tokens")
    MAX_TOKENS_PER_S = max(count(r["text"]) / ftkit.duration(r) for r in splits["train"])
    print(f"{name}: {sum(p.numel() for p in model.parameters()) / 1e6:.0f}M parameters, densest train "
          f"label {MAX_TOKENS_PER_S:.1f} tokens/s")
    return model


def retry_one(row):
    cap = min(MAX_NEW_TOKENS, math.ceil(MAX_TOKENS_PER_S * ftkit.duration(row)) + 2)
    return transcribe([row], max_new_tokens=cap, **RETRY)[0]


def decoder():
    return ftkit.RetryLoops(transcribe, retry_one)


def optimizer_for(model, name):
    return torch.optim.AdamW(model.parameters(), lr=LR[name], weight_decay=0.0, fused=DEVICE == "cuda")


def probe_budget(model, optimizer, name):
    """The largest micro-batch at the longest train clip carrying the longest transcript, with the
    optimizer state allocated. Whisper pays for 30 s whatever a clip's length, so for it the
    budget is a clip count, not seconds of audio."""
    longest = max(splits["train"], key=ftkit.duration)
    wordiest = max(splits["train"], key=lambda r: len(r["text"]))
    worst = {**longest, "text": wordiest["text"]}

    def probe_step(n):
        b = collate([worst] * n)
        with torch.autocast(DEVICE, dtype=torch.bfloat16):
            loss, _ = loss_fn(model, b)
        loss.backward()

    model.train()
    ftkit.init_optimizer_state(model, optimizer)
    n = ftkit.probe_max_items(probe_step, 1, 256, [p for p in model.parameters() if p.requires_grad])
    if name == "whisper":
        budget, items = float("inf"), max(1, int(n * PROBE_FRACTION))
    else:
        budget, items = n * ftkit.duration(longest) * PROBE_FRACTION, 256
    print(f"largest micro-batch at {ftkit.duration(longest):.0f} s: {n} clips -> "
          f"{items} clips or {budget:.0f} s of audio per micro-batch")
    return budget, items

## Training and scoring helpers

The probe finds the largest micro-batch at the longest train clip with the longest target, with
the optimizer state already allocated, per student. Scoring decodes gold and val with the best
weights and writes: overall and per clip class scores with S/D/I, and the paired difference to
Flex (student minus Flex, in WER points, 95% CI over episodes) overall and per class value of
`REPORT_KEYS`.

In [ ]:
import datetime as dt
import gc
import traceback

api = HfApi(token=os.environ["HF_TOKEN"])
api.create_repo(OUT_REPO, repo_type="model", private=True, exist_ok=True)
def decode(rows):
    # `decoder()` is the student cell's: greedy for a transducer, greedy + loop retry for the others
    d = decoder()
    texts, compute = ftkit.transcribe_rows(rows, d, budget_s=EVAL_BUDGET_S, max_items=EVAL_ITEMS,
                                           pad_to_s=PAD_TO_S)
    return texts, compute, getattr(d, "log", [])


def evaluate(model, rows=None):
    rows = rows or splits["val"]
    texts, _, _ = decode(rows)
    return score([r["text"] for r in rows], texts)


def paired(rows, refs, a, b, keys):
    # WER(b) - WER(a) in points [95% CI, episodes resampled], overall and per class value
    ca, cb, eps = score.per_clip(refs, a), score.per_clip(refs, b), [r["episode_id"] for r in rows]
    out = {"all": sweep.paired_bootstrap(ca, cb, eps)}
    for key in keys:
        for value in sorted({str((r.get("classes") or {}).get(key)) for r in rows} - {"None"}):
            idx = [i for i, r in enumerate(rows) if str((r.get("classes") or {}).get(key)) == value]
            out[f"{key}={value}"] = sweep.paired_bootstrap([ca[i] for i in idx], [cb[i] for i in idx],
                                                           [eps[i] for i in idx])
    return out


def score_and_write(out, run, name, history):
    model.eval()
    results = {}
    for split in ("gold", "val"):
        rows = splits[split]
        refs = [r["text"] for r in rows]
        texts, compute, log = decode(rows)
        m = score(refs, texts)
        m["rtf"] = sum(compute) / sum(ftkit.duration(r) for r in rows)
        m["retried"] = [{"segment_id": s, "first": f, "retry": t} for s, f, t in log]
        m["by_class"] = distill.by_class(rows, refs, texts, score)
        m["vs_flex"] = paired(rows, refs, flex[split], texts, REPORT_KEYS)
        results[split] = m
        ftkit.write_hyps(out / "harness" / f"{split}.jsonl", rows, texts, compute)
        (out / f"{split}_metrics.json").write_text(json.dumps(m, indent=1, ensure_ascii=False))
    evals = [h for h in history if "val_wer" in h]
    gold, val = results["gold"], results["val"]
    card = {
        "name": f"{name} FT (distil step 0)",
        "created_at": dt.datetime.now(dt.UTC).isoformat(timespec="seconds"),
        "description": f"{RUN_PREFIX}: {name} fine-tuned on the verified train labels only, "
                       f"{CARD_NOTE} (roadmap §B step 0)",
        "architecture": ARCHITECTURE[name],
        "base_model": BASE_MODEL[name],
        "decoder": DECODER_NAME,
        "run_name": run,
        "epochs": EPOCHS[name],
        "best_epoch": min(evals, key=lambda h: h["val_wer"])["epoch"] if evals else None,
        "lr": lr_card(name),
        "val_wer": val["wer"],
        "gold_wer": gold["wer"],
        "val_sid": {k: val[k] for k in ("sub", "del", "ins")},
        "gold_sid": {k: gold[k] for k in ("sub", "del", "ins")},
        "train_export": {k: export.get(k) for k in ("exported_at", "git_commit", "row_count",
                                                    "normalization_version", "label_version")},
    }
    (out / "harness" / "model_card.json").write_text(json.dumps(card, indent=1, ensure_ascii=False))
    keep = ("wer", "raw_wer", "cer", "sub", "del", "ins", "loops", "clips", "rtf", "vs_flex")
    row = {"run_name": run, "student": name, "best_epoch": card["best_epoch"],
           **{split: {k: results[split][k] for k in keep} for split in results},
           "gold_by_class": {k: gold["by_class"].get(k) for k in REPORT_KEYS}}
    (out / "result.json").write_text(json.dumps(row, indent=1, ensure_ascii=False))
    for split in ("val", "gold"):
        d, lo, hi = results[split]["vs_flex"]["all"]
        print(f"{run} {split}: WER {results[split]['wer']:.2f} ({ftkit.sid(results[split])}) | "
              f"minus Flex {d:+.2f} [{lo:+.2f}, {hi:+.2f}]")
    return row


def uploaded_result(run):
    remote = f"{RUN_PREFIX}/{run}/result.json"
    if not api.file_exists(OUT_REPO, remote):
        return None
    return json.loads(Path(hf_hub_download(OUT_REPO, remote, token=os.environ["HF_TOKEN"])).read_text())


def free_model():
    for var in ("optimizer", "model"):
        globals().pop(var, None)
    gc.collect()
    torch.cuda.empty_cache()


def upload(out, run, message, **kwargs):
    api.upload_folder(repo_id=OUT_REPO, folder_path=str(out), path_in_repo=f"{RUN_PREFIX}/{run}",
                      commit_message=f"{run}: {message}", **kwargs)


# Train, score and upload one student, unless OUT_REPO already has its result. A failure frees
# the GPU before it is raised: the traceback is printed as text, so no frame keeps the model or its
# optimizer state alive, and the next student's cell can run.
def train_student(name):
    run = f"{RUN_PREFIX}-{name}"
    if uploaded_result(run) is not None:
        print(f"{run}: already in {OUT_REPO}, skipped")
        return
    failure, stopped = None, False
    try:
        train_and_score(name, run)
    except BaseException as e:
        failure, stopped = traceback.format_exc(), isinstance(e, KeyboardInterrupt)
    finally:
        free_model()
    if stopped:
        raise KeyboardInterrupt(f"{run} stopped by hand outside training; the GPU is freed")
    if failure:
        print(failure)
        raise RuntimeError(f"{run} failed (traceback above); the GPU is freed for the next student")


monitor = ftkit.GpuMonitor().start()

## Train, score and upload each student

Each student has its own cell below, so a crash takes down one student, not the rest. Per student:
fresh weights, the batch probe, the speed check (it projects the run and gives val WER before
training), training with early stopping on val, then gold and val decoded with the best weights and
scored. **The best weights go to `OUT_REPO` as soon as training ends**, before scoring; the scores
follow when scoring finishes. A student whose result is already there is skipped, so after a
kernel restart, re-run the cells above and then every student cell. **Read the speed check's line
before walking away**: low GPU utilisation or a projection of many hours means a setting needs
changing.

In [ ]:
def train_and_score(name, run):
    OUT = OUT_ROOT / run
    OUT.mkdir(parents=True, exist_ok=True)
    print(f"\n=== {run}")
    load_student(name)
    optimizer = optimizer_for(model, name)
    budget, items = probe_budget(model, optimizer, name)

    def make_batches(epoch, budget=budget, items=items):
        return ftkit.bucket_batches(splits["train"], budget_s=budget, max_items=items, pad_to_s=PAD_TO_S,
                                    shuffle=True, seed=epoch)

    # Val before training on every 12th val clip, as in Distill.ipynb; the full-val time follows
    sample = splits["val"][::12]
    speed = ftkit.speed_check(model, rows=splits["train"], batches=make_batches(0), collate=collate,
                              loss_fn=loss_fn, evaluate=lambda m: evaluate(m, sample), val_rows=sample,
                              gold_rows=splits["gold"], epochs=EPOCHS[name], monitor=monitor)
    full_val_min = speed["val_s"] * len(splits["val"]) / len(sample) / 60
    print(f"a full val pass takes about {full_val_min:.1f} min, {EPOCHS[name]} of them at most "
          f"{EPOCHS[name] * full_val_min / 60:.1f} h on top of training")
    (OUT / "speed_check.json").write_text(json.dumps(speed, indent=1))

    def save_best(model, out=OUT):
        (out / "best").mkdir(exist_ok=True)
        save_to(model, out / "best")

    cfg = ftkit.TrainConfig(name=run, out=str(OUT), epochs=EPOCHS[name], lr=LR[name], warmup_frac=WARMUP,
                            effective_s=EFFECTIVE_S, patience=PATIENCE, min_delta=MIN_DELTA)
    result = ftkit.train(model, cfg=cfg, rows=splits["train"], make_batches=make_batches, collate=collate,
                         loss_fn=loss_fn, evaluate=evaluate, save_best=save_best, optimizer=optimizer,
                         monitor=monitor)
    upload(OUT, run, "best weights (step 0, before scoring)")
    score_and_write(OUT, run, name, result["history"])
    upload(OUT, run, "step 0 scores", ignore_patterns=["best/*"])

### qwen

In [ ]:
train_student("qwen")

### whisper

In [ ]:
train_student("whisper")

## Report

Each student against Flex on the same clips, folded WER with S/D/I, then the paired difference
(student minus Flex) per clip class. **The success bar** (roadmap §B) is a student whose interval
contains zero, or lies below it, on gold and val, class by class. Gold is the held-out score:
nothing here was chosen on it.

In [ ]:
results = []
for name in STUDENTS:
    if (row := uploaded_result(f"{RUN_PREFIX}-{name}")) is None:
        print(f"{name}: no result in {OUT_REPO}, left out of the report")
    else:
        results.append(row)


def line(m):
    return f"{m['wer']:6.2f} ({ftkit.sid(m)})"


print(f"{'system':<16}{'epoch':>6}  {'val':<34}{'gold':<34}{'gold raw':>9}{'gold RTF':>10}")
print(f"{'Flex p00-s0':<16}{'-':>6}  {line(flex_scores['val']):<34}{line(flex_scores['gold']):<34}"
      f"{flex_scores['gold']['raw_wer']:9.2f}{'-':>10}")
for r in results:
    print(f"{r['student']:<16}{str(r['best_epoch']):>6}  {line(r['val']):<34}{line(r['gold']):<34}"
          f"{r['gold']['raw_wer']:9.2f}{r['gold']['rtf']:10.4f}")
for r in results:
    print(f"\n{r['student']} minus Flex, WER points [95% CI, episodes resampled]:")
    for split in ("val", "gold"):
        cells = [f"{k} {d:+.2f} [{lo:+.2f},{hi:+.2f}]" for k, (d, lo, hi) in r[split]["vs_flex"].items()]
        print(f"  {split}: " + "\n        ".join(cells))
summary = {"run_prefix": RUN_PREFIX, "flex_reference": FLEX_REFERENCE,
           "flex": {k: {x: v[x] for x in ("wer", "raw_wer", "cer", "sub", "del", "ins", "clips")}
                    for k, v in flex_scores.items()},
           "students": results}
(OUT_ROOT / f"summary-{NOTEBOOK}.json").write_text(json.dumps(summary, indent=1, ensure_ascii=False))
api.upload_file(path_or_fileobj=str(OUT_ROOT / f"summary-{NOTEBOOK}.json"),
                path_in_repo=f"{RUN_PREFIX}/summary-{NOTEBOOK}.json",
                repo_id=OUT_REPO, commit_message=f"{RUN_PREFIX}: summary")
print("\nuploaded", f"https://huggingface.co/{OUT_REPO}/tree/main/{RUN_PREFIX}")